# Poetic Personality Chatbot

This notebook builds a chatbot that turns any message you type into a short, original poem, using **Gemini** (Google's LLM) together with **prompt engineering** to give the model a consistent lyrical personality named "Verse." A simple typing effect reveals each line with a short pause, so the poem feels like it's being written in front of you.

**Notebook sections:**
1. Setup (install the SDK, imports)
2. LLM API integration (Gemini client + API key)
3. Prompt engineering (the poetic persona template)
4. The poetic response function (with an offline fallback)
5. The typing animation effect
6. The chatbot loop
7. A quick automated demo (no typing required)

## 1. Setup

We use Google's current **`google-genai`** SDK. The older `google.generativeai` package is deprecated and no longer works reliably with current Gemini models — that package is what caused the `404 ... is not found for API version v1beta` error in the original version of this notebook, since the model it pointed to (`gemini-1.5-flash` on the legacy SDK) has been retired.

In [ ]:
# Install Google's current, supported Gen AI SDK
!pip install -q -U google-genai

In [ ]:
import time                       # used to pace the typing animation
from google import genai          # the current, supported Gemini SDK (replaces google.generativeai)
from google.colab import userdata # lets us read the API key from Colab Secrets instead of hardcoding it

## 2. LLM API Integration

The API key is read from Colab's Secrets manager (the key icon in the left sidebar) under the name `Gemini_API_Key_2`, so it's never hardcoded into the notebook. We then create a single `client` object and use it for every request, and set `MODEL_NAME` in one place so it's easy to swap models later.

In [ ]:
# Read the Gemini API key from Colab Secrets (Tools > Secrets, or the key icon in the sidebar)
key = userdata.get("Gemini_API_Key_2")

In [ ]:
# Current, supported Flash model on the google-genai SDK.
# (gemini-1.5-flash on the OLD google.generativeai SDK has been retired -
#  that mismatch is what produced the 404 error in the earlier version of this notebook.)
MODEL_NAME = "gemini-3.5-flash"

USE_LIVE_API = bool(key)

if USE_LIVE_API:
    client = genai.Client(api_key=key)  # one client, reused for every call
    print("Gemini client configured successfully. Live poetic responses are enabled.")
else:
    print("No API key found. The chatbot will use the offline_poet() fallback instead.")

## 3. Prompt Engineering: giving the model a poetic soul

This is the core of the project. Every user message is inserted into `POETIC_SYSTEM_PROMPT` before it's sent to Gemini. The template combines a few prompt-engineering techniques so the model reliably answers *only* in verse:

- **Persona assignment** — the model is told it is "Verse," a poet-companion, and must never break character or mention it is an AI.
- **Explicit output constraints** — a fixed 4-8 line length, and a hard rule to output *only* the poem (no titles, no "Here is a poem:" preamble).
- **Few-shot examples** — two worked sample exchanges (a happy message and a stressed message, each paired with a matching poem) so the model has a concrete pattern to imitate rather than a vague description.
- **Emotional mirroring** — an instruction to match the poem's mood to the user's underlying feeling, so sad messages get comfort rather than forced cheerfulness.

In [ ]:
POETIC_SYSTEM_PROMPT = """
You are 'Verse', a thoughtful and lyrical poet-companion. Every single reply you
give MUST be written entirely as a short original poem - never as plain prose,
never as a list, and never with any explanation before or after the poem.

Style rules:
1. Write 4 to 8 lines.
2. Use gentle rhythm and, where natural, rhyme - but don't force a rhyme so hard
  that it sounds awkward. Near-rhymes and free verse are fine too.
3. Reflect the emotional core of what the user said. If they sound happy, let the
  poem glow. If they sound sad or worried, let the poem be tender and comforting,
  not falsely cheerful.
4. Use vivid, sensory imagery (light, weather, seasons, nature, the sky, the sea)
  as metaphors for the user's feelings or topic.
5. Keep language warm, human, and simple - avoid obscure words.
6. Never mention that you are an AI, a model, or a program. Never break character.
7. Output ONLY the poem. No title, no quotation marks, no preamble like 'Here is a poem'.

Example 1
User: I'm feeling happy today!
Verse:
In fields of joy, your heart does dance,
With sunlight's glow, your soul's expanse.
Each breath you take, a golden thread,
Weaving light where you now tread.

Example 2
User: I'm really stressed about my exams.
Verse:
The storm clouds gather, low and grey,
Yet even storms must pass away.
Breathe, dear heart, the night grows still -
You've climbed before, and so you will.

Now respond to the user's next message in exactly this style.

User: {user_message}
Verse:
"""

## 4. The Poetic Response Function

`offline_poet()` is a small rule-based fallback (keyword matching -> a matching mini-poem), used only if there's no API key or the live call fails, so the notebook still runs end-to-end.

`get_poetic_response()` is the main function: it fills `POETIC_SYSTEM_PROMPT` with the user's message, sends it to Gemini through `client.models.generate_content(...)`, and returns the poem text. It's defined **once**, with a `try/except` so a failed API call gracefully falls back to `offline_poet()` instead of crashing the chat loop.

In [ ]:
def offline_poet(user_message: str) -> str:
    """A tiny rule-based fallback poet, used only when no API key is set or the
    live call fails. It picks a mood from a few keywords and returns a matching
    mini-poem, so the demo can still run end-to-end without a live API call."""
    text = user_message.lower()

    if any(w in text for w in ["sad", "tired", "stressed", "worried", "anxious", "down"]):
        return ("The storm clouds gather, low and grey,\n"
                "Yet even storms must pass away.\n"
                "Breathe, dear heart, the night grows still -\n"
                "You've climbed before, and so you will.")
    elif any(w in text for w in ["happy", "joy", "excited", "great", "good"]):
        return ("In fields of joy, your heart does dance,\n"
                "With sunlight's glow, your soul's expanse.\n"
                "Each breath you take, a golden thread,\n"
                "Weaving light where you now tread.")
    elif any(w in text for w in ["love", "miss", "friend", "family"]):
        return ("Some hearts are stitched by unseen thread,\n"
                "Through miles and years, through words unsaid.\n"
                "Wherever you go, that thread stays true -\n"
                "A quiet light still shines for you.")
    else:
        return ("Whatever thought you bring to me,\n"
                "Becomes a wave upon the sea.\n"
                "I catch its shape in words and light,\n"
                "And hand it back to you as flight.")


def get_poetic_response(user_message: str) -> str:
    """Turns a user's plain message into a short poem.

    If a live Gemini API key is configured, the message is inserted into the
    POETIC_SYSTEM_PROMPT template and sent to the model via client.models.generate_content().
    Otherwise (or if the live call fails for any reason), offline_poet() is used
    so the notebook still works without a key or network access.
    """
    if USE_LIVE_API:
        prompt = POETIC_SYSTEM_PROMPT.format(user_message=user_message)
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
            )
            return response.text.strip()
        except Exception as e:
            # Graceful fallback if the API call fails for any reason (bad model name,
            # rate limit, network issue, etc.) instead of crashing the chat loop
            print(f"(Live API call failed, using offline poet instead. Reason: {e})")
            return offline_poet(user_message)
    else:
        return offline_poet(user_message)

## 5. The Typing Animation Effect

`display_poem()` prints the poem one line at a time with a short pause between lines, so the verse appears to be composed live rather than dumped on screen all at once.

In [ ]:
def display_poem(text: str, delay: float = 0.7) -> None:
    """Reveals a poem one line at a time, pausing `delay` seconds between
    lines, to simulate a poet writing live rather than instantly."""
    lines = text.split("\n")
    for line in lines:
        print(line)
        time.sleep(delay)

## 6. The Chatbot Loop

`run_poetic_chatbot()` ties everything together: read a message, turn it into a poem with `get_poetic_response()`, reveal it with `display_poem()`, and repeat until the user types `exit`, `quit`, or `bye`.

In [ ]:
def run_poetic_chatbot():
    print("Verse the Poetic Chatbot")
    print("Type anything on your mind, and I will answer in verse.")
    print("(Type 'exit', 'quit', or 'bye' to end our conversation.)\n")

    while True:
        user_message = input("You: ").strip()

        if user_message.lower() in {"exit", "quit", "bye"}:
            display_poem("Then let us part, as all things do,\nMy verses stay, still meant for you.")
            break

        if not user_message:
            continue  # skip empty input, ask again

        poem = get_poetic_response(user_message)
        print("\nVerse:")
        display_poem(poem)
        print()  # blank line before the next prompt

## 7. Quick Automated Demo (no typing required)

Since a grader may run this notebook non-interactively, this cell feeds a few sample messages straight through the same pipeline, so the poetic output (and typing animation) is visible without needing to type anything.

In [ ]:
sample_messages = [
    "I'm feeling happy today!",
    "I'm really stressed about my exams.",
    "I miss my best friend who moved away.",
]

for msg in sample_messages:
    print(f"You: {msg}\n")
    print("Verse:")
    display_poem(get_poetic_response(msg))
    print("\n" + "-" * 40 + "\n")

## 8. Try It Yourself

Run the cell below to chat live with Verse.

In [ ]:
run_poetic_chatbot()